In [1]:
from functions import *

In [2]:
a_vec = [763, 679, 397, 61, 697, 373, 
         289, 257, 625, 41, 193, 449]
b_vec = [435, 69, 330, 18, 612, 246, 
         496, 640, 200, 524, 672, 672] 

In [3]:
G = generate_g(a_vec)
all_indices = [i for i in range(l_h**2)]
gb_indices = [3, 8]
ga_indices = [i for i in all_indices if i not in gb_indices]
Ga = G.extract(ga_indices, list(range(G.cols)))
Gb = G.extract(gb_indices, list(range(G.cols)))
basis_a = solve_modular_kernel(Ga, P)
V = Matrix.hstack(*basis_a)

In [4]:
# 1. b_vec を行列形式に変換
b_mat = Matrix(b_vec)

# 2. V * x = b を有理数体（Q）上で解く
# V は main.ipynb で定義された Matrix.hstack(*basis_a)
try:
    # V が正則であれば x = V^-1 * b が求まる
    x_rational = V.solve(b_mat)

    # 3. 有理数の解 a/b を整数 a * inv(b, P) (mod P) に変換する関数
    def to_mod_p(val, p):
        num, den = val.as_numer_denom()
        # Python 3.8+ の pow(den, -1, p) はモジュラ逆数を計算する
        return (int(num) * pow(int(den), -1, p)) % p

    # 各要素に適用
    coefficients = x_rational.applyfunc(lambda v: to_mod_p(v, P))

    print("線形結合の係数ベクトル x:")
    display(coefficients)
    
    # 検算: V * x % P が b_vec と一致するか確認
    check_val = (V * coefficients).applyfunc(lambda x: x % P)
    if check_val == b_mat.applyfunc(lambda x: x % P):
        print("検算成功: 一致しました。")
    else:
        print("警告: 検算に失敗しました。")

except Exception as e:
    print(f"解を求めることができませんでした: {e}")

線形結合の係数ベクトル x:


Matrix([
[  3],
[709],
[689],
[746],
[762],
[732],
[ 68],
[ 84],
[565],
[557],
[744],
[346]])

検算成功: 一致しました。


In [5]:
cycles = generate_cycles(6)
h_x, h_z = generate_h_xz()
constraints = generate_constraints(cycles, a_vec, h_x, h_z)

In [6]:
# 全ての禁止ベクトル（法ベクトル）を個別にリスト化する
unique_forbidden_vectors = []
seen_vectors = set()

# 1. 条件B (潜在部の非可換性) からの制約 r_i
for i in range(Gb.rows):
    c_prime = (Gb.row(i) * V).applyfunc(lambda x: x % P)
    c_tuple = tuple(c_prime)
    if c_tuple not in seen_vectors:
        unique_forbidden_vectors.append(c_prime.T) # 列ベクトルとして保存
        seen_vectors.add(c_tuple)

# 2. 条件C (短いサイクルの回避) からの制約 c_prime
for c in constraints:
    c_prime = (Matrix([c]) * V).applyfunc(lambda x: x % P)
    c_tuple = tuple(c_prime)
    if c_tuple not in seen_vectors:
        unique_forbidden_vectors.append(c_prime.T)
        seen_vectors.add(c_tuple)

print(f"個別に回避すべき禁止制約（超平面）の数: {len(unique_forbidden_vectors)}")

個別に回避すべき禁止制約（超平面）の数: 312


In [7]:
def is_in_general_solution(x_vec, forbidden_vectors, p):
    """
    x_vec がすべての禁止超平面 r_i^T * x = 0 (mod p) を避けているか判定する。
    """
    # ベクトル形式を整える
    x_mat = Matrix(x_vec)
    
    for r in forbidden_vectors:
        # 内積が 0 (mod P) になったらその禁止領域に含まれている
        if (r.T * x_mat)[0] % p == 0:
            return False
    return True

# すでに見つけている特殊解 coefficients (x0) の妥当性を再確認
if is_in_general_solution(coefficients, unique_forbidden_vectors, P):
    print("特殊解 x0 は一般解の条件をすべて満たしています。")

特殊解 x0 は一般解の条件をすべて満たしています。


In [8]:
# 1. 既知の特殊解を分解する
x0 = coefficients # Matrix([3, 709, ...])
x0_3 = x0.applyfunc(lambda x: x % 3)
x0_256 = x0.applyfunc(lambda x: x % 256)

# 2. 法 3 の空間における「安全な」一般解の探索
# da=12 なので 3^12 は全探索も可能だが、特殊解周辺を効率よく調べる
def find_safe_mod3_component(x0_3, forbidden_vectors, count=3):
    safe_3 = []
    da = x0_3.rows
    # 近傍（距離1の範囲）を探索
    for delta in product([-1, 0, 1], repeat=da):
        if sum(abs(d) for d in delta) > 2: continue # 探索範囲を絞る
        
        dx = Matrix(delta)
        candidate = (x0_3 + dx).applyfunc(lambda x: x % 3)
        
        # mod 3 で避けている制約の数をカウント
        hit_count = 0
        for r in forbidden_vectors:
            if (r.T * candidate)[0] % 3 != 0:
                hit_count += 1
        
        # 評価の高い（より多く避けている）成分を保持
        safe_3.append((hit_count, candidate))
        
    safe_3.sort(key=lambda x: x[0], reverse=True)
    return [s[1] for s in safe_3[:count]]

# 法 3 で安全な成分を取得
safe_x3_list = find_safe_mod3_component(x0_3, unique_forbidden_vectors)

In [9]:
final_general_solutions = []
for x3 in safe_x3_list:
    # x3 (mod 3) と x0_256 (mod 256) を合成
    x_gen = crt_combine_matrix(x3, x0_256)
    
    # 全体の法 P=768 で制約をチェック
    if is_in_general_solution(x_gen, unique_forbidden_vectors, P):
        final_general_solutions.append(x_gen)

print(f"CRT分解により、代数的に保証された {len(final_general_solutions)} 個の一般解を生成しました。")

CRT分解により、代数的に保証された 3 個の一般解を生成しました。


In [11]:
# 生成された一般解の一つを b_vec に戻して確認
if final_general_solutions:
    for sample_x in final_general_solutions:
        # b = V * x (mod P)
        b_vec = (V * sample_x).applyfunc(lambda val: val % P)
        
        print("b_vec:", list(b_vec))
        print("check:", check(a_vec, b_vec), '\n')



b_vec: [435, 69, 330, 18, 612, 246, 752, 640, 712, 524, 672, 672]
check: True 

b_vec: [435, 69, 330, 18, 612, 246, 496, 128, 200, 268, 672, 672]
check: True 

b_vec: [435, 69, 330, 18, 612, 246, 496, 640, 456, 524, 416, 672]
check: True 

